In [2]:
import subprocess
import sys

# -----------------------------
# Install Required Libraries
# -----------------------------

libraries = [
    "numpy",
    "scikit-learn",
    "tensorflow",
    "scikeras",
    "matplotlib",
    "pillow"
]

print("Installing required libraries...\n")

for lib in libraries:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", lib])
        print(f"{lib} installed successfully!")
    except Exception as e:
        print(f"Failed to install {lib}: {e}")

print("\nAll required libraries installed successfully!\n")


# -----------------------------
# Imports for Your Code
# -----------------------------

import numpy as np

from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score
from sklearn.base import BaseEstimator, ClassifierMixin

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Correct wrapper for sklearn + keras
from scikeras.wrappers import KerasClassifier

print("All libraries imported successfully!")

Installing required libraries...

numpy installed successfully!
scikit-learn installed successfully!
tensorflow installed successfully!
scikeras installed successfully!
matplotlib installed successfully!
pillow installed successfully!

All required libraries installed successfully!

All libraries imported successfully!


In [3]:
# Define image dimensions and batch size
img_height, img_width = 128, 128
batch_size = 32
# Define directories for training and testing data
train_data_dir = "../dataset/train"
test_data_dir = "../dataset/test"

In [4]:
# Data augmentation for training images
train_datagen = ImageDataGenerator(
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    rescale=1./255  # Normalize pixel values
)

# Data augmentation for testing images (only rescale)
test_datagen = ImageDataGenerator(rescale=1./255)


In [5]:
# Create data generators for training and testing
train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False  # No need to shuffle for evaluation
)



Found 1174 images belonging to 14 classes.
Found 428 images belonging to 14 classes.


In [6]:
# Define the ResNet50 base model
def create_resnet_model():
    resnet_model = ResNet50(
        input_shape=(img_height, img_width, 3),  # Adjust input shape
        include_top=False,  # Exclude the fully-connected layers
        weights='imagenet'  # Pre-trained on ImageNet
    )
    resnet_model.trainable = False

    model = Sequential([
        resnet_model,
        GlobalAveragePooling2D(),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(14, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    print(model.summary())
    return model






In [7]:
#MODEL fit and training
model = create_resnet_model()
model.fit(train_generator, epochs=15, validation_data=test_generator)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 27s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       131,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 14)             │           910 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,720,014 (90.48 MB)

 Trainable params: 132,174 (516.30 KB)

 Non-trainable params: 23,587,840 (89.98 MB)

None
Epoch 1/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 27s 581ms/step - accuracy: 0.1516 - loss: 2.4637 - val_accuracy: 0.0374 - val_loss: 3.0548
Epoch 2/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 19s 511ms/step - accuracy: 0.2308 - loss: 2.1994 - val_accuracy: 0.0374 - val_loss: 2.9150
Epoch 3/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 15s 410ms/step - accuracy: 0.2632 - loss: 2.0848 - val_accuracy: 0.0631 - val_loss: 2.7427
Epoch 4/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 15s 405ms/step - accuracy: 0.3024 - loss: 2.0112 - val_accuracy: 0.0771 - val_loss: 2.6470
Epoch 5/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 16s 421ms/step - accuracy: 0.3032 - loss: 1.9682 - val_accuracy: 0.0958 - val_loss: 2.5428
Epoch 6/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 24s 641ms/step - accuracy: 0.3169 - loss: 1.9074 - val_accuracy: 0.0397 - val_loss: 2.6931
Epoch 7/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 19s 514ms/step - accuracy: 0.3416 - loss: 1.8577 - val_accuracy: 0.0958 - val_loss: 2.5073
Epoch 8/15
37/37 ━━━━━━━━━━━━━━━━━━━━ 16s 429ms/step - accuracy: 0.3560 - loss: 1.8164 - val

In [9]:
loss, accuracy = model.evaluate(test_generator)
print(f"Accuracy: {accuracy*100:.2f}%")

14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 296ms/step - accuracy: 0.1332 - loss: 4.4124
Accuracy: 13.32%


In [10]:
# Save the trained model
model.save("../model_saved_files/ResNet.h5")
print("Model saved successfully!")

Model saved successfully!
